# External `dataV2` benchmark trên Google Colab

Notebook này chạy một sanity/robustness analysis mô tả trên bộ **geographically anchored semi-synthetic** `dataV2`. OSM, Copernicus và WorldPop chỉ là bối cảnh địa lý; incident, report, duplicate, fake campaign và nhãn đánh giá là mô phỏng. Kết quả không phải real-world validation và không được đưa ngược vào confirmation v2 một cách cơ học.

Inference chỉ đọc `algorithm_input.json`. `ground_truth.json`, `observable_reports.json`, `latent_incidents.json` và `run_manifest.json` được dùng sau đó cho join/audit/evaluation. Bốn configuration đã khóa được kiểm tra nguyên vẹn; notebook không tuning.

## Cách chạy nhanh

1. Commit và push code + `demo/dataV2`, lấy SHA 40 ký tự.
2. Dán SHA vào cell cấu hình bên dưới và chạy tuần tự từ đầu.
3. Chạy `RUN_MODE='smoke'` trước để kiểm tra môi trường; sau đó đổi thành `full` và chạy setup để tạo artifact directory mới (smoke không tạo batch checkpoint 80-run).
4. Nếu Colab mất kết nối sau khi full đã bắt đầu, upload/unpack đúng artifact directory, đặt `ARTIFACT_DIR_OVERRIDE` và chạy lại: batch runner sẽ resume các checkpoint có provenance khớp.
5. Cell cuối cho phép tải ZIP và notebook đã thực thi. Không chỉnh tay các JSON/CSV sau khi tải về.

Runtime CPU là đủ; GPU không được dùng. Full batch gồm 80 run × 4 method, có thể mất vài phút tùy runtime. Khi mất kết nối, upload/unpack checkpoint vào `/content`, đặt `ARTIFACT_DIR_OVERRIDE` trỏ đúng thư mục đó rồi chạy lại setup; runner sẽ từ chối resume nếu provenance bị drift.

## 1 — Tham số người chạy

Cell này là nơi duy nhất cần sửa trước khi chạy. `REPO_COMMIT_SHA` bắt buộc là commit bất biến; notebook sẽ dừng nếu để placeholder hoặc dùng branch/tag. `RANDOM_STATE` dùng cho các method đã khóa; `BOOTSTRAP_SEED` chỉ dùng cho CI mô tả.

In [ ]:
REPO_URL = 'https://github.com/ngthtrong/nckh2.git'
REPO_COMMIT_SHA = 'PASTE_40_CHAR_COMMIT_SHA_HERE'
RANDOM_STATE = 42
BOOTSTRAP_SEED = 20260812
N_BOOTSTRAP = 10_000
RUN_MODE = 'smoke'  # đổi thành 'full' sau khi smoke pass
ARTIFACT_DIR_OVERRIDE = None  # ví dụ: '/content/external_sanity_20260812T120000Z_<sha>' để resume
EXPECTED_RUNS = 1 if RUN_MODE == 'smoke' else 80
assert RUN_MODE in {'smoke', 'full'}
print({'run_mode': RUN_MODE, 'expected_runs': EXPECTED_RUNS, 'random_state': RANDOM_STATE, 'bootstrap_seed': BOOTSTRAP_SEED})

## 2 — Kiểm tra Colab và Python

Dữ liệu và code được kiểm thử với CPython 3.12. Cell này in thông tin tài nguyên để gắn vào provenance và dừng sớm khi runtime không tương thích. Colab đã có runtime Python 3.12, nhưng không nên giả định phiên bản mà không kiểm tra.

In [ ]:
import os, platform, shutil, sys
from pathlib import Path

if sys.version_info[:2] != (3, 12):
    raise RuntimeError(f'Yêu cầu CPython 3.12, nhận được {sys.version}')
print('Python:', sys.version)
print('Platform:', platform.platform())
print('CPU:', os.cpu_count())
print('RAM/disk snapshot:', shutil.disk_usage('/content'))

## 3 — Clone snapshot bất biến

Cell này clone repository vào `/content/nckh2`, checkout đúng SHA và xác minh `HEAD`. Nếu thư mục đã tồn tại nhưng SHA khác, cell dừng thay vì âm thầm dùng code khác.

In [ ]:
import re, subprocess

if not re.fullmatch(r'[0-9a-fA-F]{40}', REPO_COMMIT_SHA):
    raise ValueError('REPO_COMMIT_SHA phải là SHA Git 40 ký tự')
REPO_DIR = Path('/content/nckh2')
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
head = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if head.lower() != REPO_COMMIT_SHA.lower():
    subprocess.run(['git', '-C', str(REPO_DIR), 'fetch', '--all', '--tags'], check=True)
    subprocess.run(['git', '-C', str(REPO_DIR), 'checkout', '--detach', REPO_COMMIT_SHA], check=True)
    head = subprocess.check_output(['git', '-C', str(REPO_DIR), 'rev-parse', 'HEAD'], text=True).strip()
if head.lower() != REPO_COMMIT_SHA.lower():
    raise RuntimeError(f'Git HEAD mismatch: {head} != {REPO_COMMIT_SHA}')
print('Checked-out commit:', head)

## 4 — Cài môi trường khóa

Cài đúng dependency lock của repository. `pyarrow` được thêm riêng để đọc metadata parquet; nó không tham gia inference. Colab có thể đã nạp NumPy trước khi pip thay phiên bản; cell dùng probe trong một process sạch và đánh dấu dependency. Nếu pip vừa thay NumPy/Pandas đang được kernel giữ trong `sys.modules`, cell sẽ dừng rõ ràng để bạn chọn **Runtime → Restart session**, rồi chạy lại từ đầu.

In [ ]:
import hashlib, importlib.metadata as importlib_metadata, json
LOCK_SHA = hashlib.sha256((REPO_DIR / 'requirements.lock').read_bytes()).hexdigest()[:16]
DEPENDENCY_STAMP = Path('/content') / f'.datav2_dependencies_{LOCK_SHA}.json'
EXPECTED_PACKAGES = {'numpy': '2.5.1', 'pandas': '3.0.3', 'scipy': '1.18.0', 'scikit-learn': '1.9.0', 'networkx': '3.6.1', 'python-louvain': '0.16', 'pyarrow': '18.1.0'}
probe_code = "import importlib.metadata as m; expected = " + repr(EXPECTED_PACKAGES) + "; actual = {name: m.version(name) for name in expected}; print(actual); raise SystemExit(0 if actual == expected else 2)"
probe = subprocess.run([sys.executable, '-c', probe_code], text=True, capture_output=True)
probe_output = probe.stdout
if probe.returncode != 0:
    print('Installing locked dependencies:', probe.stdout, probe.stderr)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--upgrade', '--force-reinstall', '--no-cache-dir', '-q', '-r', str(REPO_DIR / 'requirements.lock'), 'pyarrow==18.1.0'], check=True)
    verify = subprocess.run([sys.executable, '-c', probe_code], text=True, capture_output=True)
    if verify.returncode != 0:
        raise RuntimeError('Dependency probe vẫn thất bại sau pip: ' + verify.stdout + verify.stderr)
    probe_output = verify.stdout
    DEPENDENCY_STAMP.write_text(json.dumps({'lock_sha256': LOCK_SHA, 'packages': EXPECTED_PACKAGES}) + '\n')
    loaded = sorted(name for name in sys.modules if name == 'numpy' or name.startswith(('numpy.', 'pandas', 'scipy', 'sklearn', 'pyarrow')))
    if loaded:
        raise RuntimeError('pip đã thay binary package nhưng kernel đã nạp: ' + ', '.join(loaded[:8]) + '. Hãy chọn Runtime → Restart session, rồi chạy lại notebook từ đầu.')
else:
    if not DEPENDENCY_STAMP.exists():
        DEPENDENCY_STAMP.write_text(json.dumps({'lock_sha256': LOCK_SHA, 'packages': EXPECTED_PACKAGES}) + '\n')
try:
    import numpy as _numpy_probe, pandas as _pandas_probe, scipy as _scipy_probe, sklearn as _sklearn_probe, pyarrow as _pyarrow_probe
except Exception as exc:
    raise RuntimeError('Kernel đang giữ NumPy/Pandas binary không nhất quán. Hãy chọn Runtime → Restart session, rồi chạy lại notebook từ đầu.') from exc
print('Dependencies ready:', probe_output.strip() or 'locked versions already available')

## 5 — Import, seed và thư mục artifact

Đặt repository lên `sys.path`, giới hạn thread để kết quả và thời gian chạy ổn định hơn, rồi tạo một thư mục artifact mới. Batch runner sẽ thêm checkpoint vào cùng thư mục này.

In [ ]:
sys.path.insert(0, str(REPO_DIR))
os.environ.setdefault('OMP_NUM_THREADS', '1')
os.environ.setdefault('OPENBLAS_NUM_THREADS', '1')
os.environ.setdefault('MKL_NUM_THREADS', '1')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from demo.v2.external_benchmark import (
    DEFAULT_SELECTION, METHOD_ORDER, aggregate_batch_results, bootstrap_paired_delta,
    dataset_audit, duplicate_audit, load_external_run, run_external_batch,
    run_external_benchmark, file_sha256,
)
DATA_ROOT = REPO_DIR / 'demo' / 'dataV2'
SELECTION_PATH = REPO_DIR / 'revision' / 'v2' / 'results' / 'calibration_selection.json'
short_sha = REPO_COMMIT_SHA[:12]
ARTIFACT_DIR = (Path(ARTIFACT_DIR_OVERRIDE) if ARTIFACT_DIR_OVERRIDE else Path('/content') / f'external_sanity_{pd.Timestamp.utcnow().strftime("%Y%m%dT%H%M%SZ")}_{short_sha}')
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('Artifact directory:', ARTIFACT_DIR)

## 6 — Ghi provenance ban đầu

Provenance được ghi trước khi chạy dữ liệu: commit, Python, hệ điều hành, resource snapshot, protocol SHA và selection SHA. File này giúp phân biệt một kết quả tái lập với một lần chạy dùng code/config khác.

In [ ]:
import datetime as dt, importlib.metadata as importlib_metadata, json
PROVENANCE = {
    'created_at_utc': dt.datetime.now(dt.timezone.utc).isoformat(),
    'git_sha': REPO_COMMIT_SHA,
    'python': sys.version,
    'platform': platform.platform(),
    'cpu_count': os.cpu_count(),
    'packages': {name: importlib_metadata.version(name) for name in ['numpy', 'pandas', 'networkx', 'scikit-learn', 'scipy', 'python-louvain', 'pyarrow']},
    'protocol_sha256': json.loads(SELECTION_PATH.read_text())['protocol_sha256'],
    'selection_sha256': file_sha256(SELECTION_PATH),
    'random_state': RANDOM_STATE,
    'bootstrap_seed': BOOTSTRAP_SEED,
    'n_bootstrap': N_BOOTSTRAP,
}
(ARTIFACT_DIR / 'environment.json').write_text(json.dumps(PROVENANCE, ensure_ascii=False, indent=2) + '\n')
print(json.dumps(PROVENANCE, ensure_ascii=False, indent=2))

## 7 — Khám phá bundle và parquet

Cell này kiểm tra quy mô công khai trước khi chạy phương pháp: số run, số file/run, kích thước file, metadata parquet và schema summary. Đây là mô tả, chưa phải quality gate.

In [ ]:
from collections import Counter
import pyarrow.parquet as pq
gold_dir = DATA_ROOT / 'gold'
run_dirs = sorted(gold_dir.glob('run_*'))
print('run count:', len(run_dirs))
print('file counts:', Counter(len(list(p.iterdir())) for p in run_dirs))
print('bundle bytes:', sum(p.stat().st_size for p in DATA_ROOT.rglob('*') if p.is_file()))
parquet_path = DATA_ROOT / 'enriched_anchors.parquet'
parquet_file = pq.ParquetFile(parquet_path)
print('parquet rows:', parquet_file.metadata.num_rows)
print('parquet columns:', parquet_file.schema.names)
print('parquet row groups:', parquet_file.metadata.num_row_groups)
parquet_table = pq.read_table(parquet_path)
parquet_nulls = {name: int(parquet_table[name].null_count) for name in parquet_table.column_names}
parquet_audit = {'n_rows': int(parquet_file.metadata.num_rows), 'columns': parquet_file.schema.names, 'null_counts': parquet_nulls, 'V_reference_null_fraction': parquet_nulls.get('V_reference', 0) / max(1, parquet_file.metadata.num_rows)}
print('V_reference null fraction:', parquet_audit['V_reference_null_fraction'])
(ARTIFACT_DIR / 'parquet_audit.json').write_text(json.dumps(parquet_audit, ensure_ascii=False, indent=2) + '\n')

## 8 — Integrity gate

Mỗi run được đọc qua `load_external_run`. Loader kiểm tra đủ năm tệp, schema, JSON finite, ID duy nhất, join theo `event_id`, duplicate lineage, latent IDs và manifest. Lỗi parse/join/schema làm cell dừng; quality finding khoa học được giữ lại ở cell sau.

In [ ]:
integrity_rows = []
for run_dir in run_dirs:
    dataset = load_external_run(run_dir)
    integrity_rows.append({
        'dataset_id': dataset.dataset_id,
        'seed': dataset.run_manifest['seed'],
        'manifest_dataset_id': dataset.run_manifest['dataset_id'],
        'n_reports': len(dataset.reports),
        'n_incidents': len(dataset.latent_incidents),
        'algorithm_sha256': dataset.algorithm_payload_sha256,
        'truth_sha256': dataset.truth_payload_sha256,
        'observable_sha256': dataset.observable_payload_sha256,
        'latent_sha256': dataset.latent_payload_sha256,
        'manifest_sha256': dataset.manifest_payload_sha256,
    })
integrity = pd.DataFrame(integrity_rows).sort_values('dataset_id')
assert len(integrity) == 80, f'expected 80 runs, got {len(integrity)}'
assert integrity['dataset_id'].tolist() == [f'run_{i:03d}' for i in range(1, 81)]
assert integrity['seed'].tolist() == list(range(1, 81))
assert (integrity['manifest_dataset_id'] == integrity['dataset_id']).all()
assert integrity['algorithm_sha256'].notna().all()
summary_df = pd.read_csv(DATA_ROOT / 'generation_summary.csv', encoding='utf-8-sig')
summary_df = summary_df.rename(columns={summary_df.columns[0]: 'dataset_id'})
assert summary_df['dataset_id'].tolist() == integrity['dataset_id'].tolist()
for field in ['n_incidents', 'n_reports']:
    assert summary_df[field].tolist() == integrity[field].tolist()
display(integrity.head())
print('Integrity gate: PASS')

## 9 — Quality audit không chặn

Các cảnh báo dưới đây là đặc tính cần báo cáo, không được sửa âm thầm: oracle fields nằm trong observable table, source identity/receipt time độc lập chưa có, numeric ID block, exact duplicate lệch fingerprint, và outcome truth không đủ cho priority/dispatch.

In [ ]:
audit_rows = []
dedup_rows = []
for run_dir in run_dirs:
    dataset = load_external_run(run_dir)
    audit = dataset_audit(dataset)
    audit['dataset_id'] = dataset.dataset_id
    audit_rows.append(audit)
    dedup = duplicate_audit(dataset)
    dedup['dataset_id'] = dataset.dataset_id
    dedup_rows.append(dedup)
audit_df = pd.DataFrame(audit_rows)
print('latent incidents:', audit_df['n_latent_incidents'].sum())
print('reports:', audit_df['n_reports'].sum())
print('runs with identifier-class block:', sum(row['identifier_audit']['truth_categories_form_disjoint_contiguous_blocks'] for row in audit_rows))
print('exact fingerprint matches:', sum(row['declared_exact_fingerprint_matches'] for row in dedup_rows))
print('near envelope matches:', sum(row['declared_near_rows_matching_near_envelope'] for row in dedup_rows))
(ARTIFACT_DIR / 'data_audit.json').write_text(json.dumps(audit_rows, ensure_ascii=False, indent=2, default=str) + '\n')
display(pd.DataFrame([{'dataset_id': r['dataset_id'], 'issues': len(r['issues']), 'missing_L': r['missing_observation_counts']['L'], 'n_claims_over_500': r['n_claims_above_priority_cap_500']} for r in audit_rows]).head())

## 10 — Mô tả phân bố report và incident

Cell này đọc raw JSON chỉ cho mục đích mô tả. Các class, số report/run, feature range, corroboration, ảnh, tỉnh/AOI và thời gian được tổng hợp để phát hiện lệch dữ liệu trước khi diễn giải metric.

In [ ]:
report_rows, incident_rows = [], []
for run_dir in run_dirs:
    observable = json.loads((run_dir / 'observable_reports.json').read_text())
    truth = {row['event_id']: row for row in json.loads((run_dir / 'ground_truth.json').read_text())['report_labels']}
    for row in observable['reports']:
        report_rows.append({'dataset_id': observable['dataset_id'], **row, 'report_class': truth[row['event_id']]['report_class'], 'duplicate_type': truth[row['event_id']]['duplicate_type']})
    latent = json.loads((run_dir / 'latent_incidents.json').read_text())
    incident_rows.extend({'dataset_id': latent['dataset_id'], **row} for row in latent['incidents'])
reports_df = pd.DataFrame(report_rows)
incidents_df = pd.DataFrame(incident_rows)
display(reports_df.groupby(['report_class', 'duplicate_type']).size().rename('n').reset_index())
display(reports_df[['flood', 'urgency', 'n_trapped', 'vulnerability', 'confidence', 'n_corrob']].describe().T)
display(reports_df.groupby('province').size().sort_values(ascending=False).rename('n').head(12))
display(incidents_df.groupby('aoi_name').size().sort_values(ascending=False).rename('n'))
reports_df['created_at_parsed'] = pd.to_datetime(reports_df['created_at'], utc=True)
display(reports_df.groupby('dataset_id')['created_at_parsed'].agg(['min', 'max']))
display(reports_df[['lat', 'lng']].describe().T)
print('empty notes:', int((reports_df['note'] == '').sum()), '/', len(reports_df))

## 11 — Minh họa không gian của một run

Hình chỉ giúp đọc cơ chế sinh dữ liệu: center incident, genuine/duplicate report và fake campaign. Không diễn giải hình này như bản đồ flood hoặc bản đồ cứu hộ thực tế.

In [ ]:
sample_id = 'run_001'
sample_dir = gold_dir / sample_id
sample_reports = pd.DataFrame(json.loads((sample_dir / 'observable_reports.json').read_text())['reports'])
sample_truth = pd.DataFrame(json.loads((sample_dir / 'ground_truth.json').read_text())['report_labels'])
sample_reports = sample_reports.merge(sample_truth[['event_id', 'report_class']], on='event_id', how='left')
sample_incidents = pd.DataFrame(json.loads((sample_dir / 'latent_incidents.json').read_text())['incidents'])
fig, ax = plt.subplots(figsize=(8, 6))
for cls, color in [('genuine', 'tab:blue'), ('duplicate', 'tab:orange'), ('coordinated_fake', 'tab:red')]:
    part = sample_reports[sample_reports['report_class'] == cls]
    ax.scatter(part['lng'], part['lat'], s=14, alpha=.55, label=cls, c=color)
ax.scatter(sample_incidents['center'].map(lambda x: x['lng']), sample_incidents['center'].map(lambda x: x['lat']), marker='x', c='black', s=70, label='latent center')
ax.set(title='run_001 — illustrative report/incident geography', xlabel='longitude', ylabel='latitude')
ax.legend()
fig.tight_layout()
fig.savefig(ARTIFACT_DIR / 'spatial_run_001.png', dpi=180)
fig.savefig(ARTIFACT_DIR / 'spatial_run_001.pdf')
plt.show()

## 12 — Bốn configuration đã khóa

Cell này chỉ đọc selection artifact để xác minh bốn method và tham số. Không có cell nào tìm kiếm grid, chọn lại cấu hình hoặc dùng ground truth để thay đổi inference.

In [ ]:
selection = json.loads(SELECTION_PATH.read_text())
selected_rows = []
for method_id, row in selection['selections'].items():
    cfg = row['configuration']
    selected_rows.append({'method_id': method_id, 'configuration_id': cfg['configuration_id'], 'operator': cfg['operator'], 'parameters': cfg['parameters']})
display(pd.DataFrame(selected_rows)[['method_id', 'configuration_id', 'operator']])
feature_notes = {
    'method.product_louvain': 'đồ thị geo-time; trọng số product của F/E/N/V và các tham số sigma/tau/alpha/beta/gamma',
    'method.additive_louvain': 'đồ thị geo-time; trọng số additive của F/E/N/V và cùng nuisance parameters',
    'method.st_dbscan': 'L và T (khoảng cách không gian/thời gian; min_samples)',
    'method.hdbscan_geo_time': 'L và T chuẩn hóa (spatial_scale/temporal_scale; min_cluster_size/min_samples)',
}
endpoint_notes = {
    'ari_linked': 'cao hơn thường tốt hơn cho agreement trên report linked',
    'false_destinations_per_100_reports': 'thấp hơn: ít destination chỉ gồm noise/fake hơn',
    'noise_rejection': 'tỷ lệ report noise bị từ chối; không thay thế fake-campaign audit',
    'review_items_per_100_reports': 'thấp hơn: ít đơn vị cần review hơn',
    'split_loss/merge_loss': 'thấp hơn: ít split/merge sai hơn',
    'max_diameter_m/singleton_rate': 'độ phân tán cực đại / tỷ lệ cluster singleton',
}
display(pd.DataFrame([{'method_id': m, 'features': feature_notes[m]} for m in METHOD_ORDER]))
display(pd.DataFrame([{'endpoint': k, 'meaning': v} for k, v in endpoint_notes.items()]))
print('Selection SHA256:', file_sha256(SELECTION_PATH))
print('Interpretation: report-level clustering/dedup diagnostics only; no priority or dispatch claim.')

## 13 — Smoke test

Smoke test chạy `run_001` qua bốn method, xác nhận output hữu hạn và đảm bảo adapter không mang `gt_cluster`/`is_fake` vào `ReportV2`. Nếu cell này lỗi, không chạy full batch.

In [ ]:
smoke_dir = gold_dir / 'run_001'
smoke = run_external_benchmark(
    reports_path=smoke_dir / 'algorithm_input.json',
    truth_path=smoke_dir / 'ground_truth.json',
    selection_path=SELECTION_PATH,
    random_state=RANDOM_STATE,
)
assert smoke['status'] == 'completed_descriptive_only'
assert set(smoke['clustering']) == set(METHOD_ORDER)
for method_id in METHOD_ORDER:
    metrics = smoke['clustering'][method_id]['metrics']
    assert all(np.isfinite(float(metrics[name])) for name in ['ari_linked', 'false_destinations_per_100_reports', 'noise_rejection'])
    coverage = smoke['clustering'][method_id]['assignment_coverage']
    assert coverage['n_reports'] == 316 == coverage['n_labels'] == coverage['n_unique_report_ids']
    assert coverage['all_reports_assigned'] is True
dataset_smoke = load_external_run(smoke_dir)
assert all(not hasattr(report, 'gt_cluster') and not hasattr(report, 'is_fake') for report in dataset_smoke.reports)
(ARTIFACT_DIR / 'smoke_run.json').write_text(json.dumps(smoke, ensure_ascii=False, indent=2, default=str) + '\n')
display(pd.DataFrame([{'method': m, **smoke['clustering'][m]['metrics']} for m in METHOD_ORDER]))

## 14 — Full batch 80 run và checkpoint

Ở chế độ `full`, runner phát hiện chính xác `run_001`–`run_080`, chạy từng run, ghi `per_run/run_NNN.json` atomic và cập nhật manifest sau mỗi run. `resume=True` chỉ dùng checkpoint nếu data/protocol/selection/Git SHA/random state khớp. Smoke mode bỏ qua cell này để tiết kiệm thời gian.

In [ ]:
if RUN_MODE == 'full':
    batch_report = run_external_batch(
        data_root=DATA_ROOT,
        output_dir=ARTIFACT_DIR,
        expected_runs=80,
        random_state=RANDOM_STATE,
        selection_path=SELECTION_PATH,
        resume=True,
        git_sha=REPO_COMMIT_SHA,
        progress_callback=lambda name, index, total, status: print(f'[{index}/{total}] {name}: {status}'),
    )
    print({k: batch_report[k] for k in ['status', 'output_dir']})
    print(batch_report['aggregate']['n_success'], 'success;', batch_report['aggregate']['n_failures'], 'failure')
else:
    batch_report = {'status': 'smoke_only', 'aggregate': {'n_success': 0, 'n_failures': 0}}
    print('Smoke mode: full batch chưa chạy; đổi RUN_MODE thành full rồi chạy lại từ cell này.')

## 15 — Coverage sau batch

Cell này không tổng hợp nếu thiếu run hoặc thiếu method. Mọi lỗi runtime phải xuất hiện trong `batch_manifest.json`/`failures.json`; không được bỏ qua bằng cách chỉ đọc các run thành công.

In [ ]:
if RUN_MODE == 'full':
    batch_manifest = json.loads((ARTIFACT_DIR / 'batch_manifest.json').read_text())
    result_files = sorted((ARTIFACT_DIR / 'per_run').glob('run_*.json'))
    assert len(result_files) == 80
    result_payloads = [json.loads(path.read_text()) for path in result_files]
    successful = [row for row in result_payloads if row.get('status') == 'completed_descriptive_only']
    for row in successful:
        assert set(row['clustering']) == set(METHOD_ORDER)
        assert all(row['inputs'].get(field) for field in ['reports_sha256', 'truth_sha256', 'observable_sha256', 'latent_sha256', 'manifest_sha256'])
        assert row['inputs']['random_state'] == RANDOM_STATE
        assert row['inputs']['selection_sha256'] == file_sha256(SELECTION_PATH)
        for method_id in METHOD_ORDER:
            coverage = row['clustering'][method_id]['assignment_coverage']
            assert coverage['n_reports'] == coverage['n_labels'] == coverage['n_unique_report_ids']
            assert coverage['all_reports_assigned'] is True
            assert all(np.isfinite(float(value)) for value in row['clustering'][method_id]['metrics'].values() if isinstance(value, (int, float)))
    print('manifest status:', batch_manifest['status'])
    print('result files:', len(result_files), 'successful:', len(successful))
else:
    result_payloads = [smoke]
    successful = result_payloads
    (ARTIFACT_DIR / 'failures.json').write_text(json.dumps({'schema_version': 'v2.external-sanity-failures.1', 'failures': []}, ensure_ascii=False, indent=2) + '\n')
    print('Coverage cell ở smoke mode chỉ kiểm tra một run.')

## 16 — Tổng hợp clustering

Các thống kê dưới đây tính theo run, không coi report là đơn vị độc lập. Báo cáo mean, SD, median, IQR, min/max cho tám endpoint; đây là descriptive summary, không phải kiểm định giả thuyết.

In [ ]:
aggregate = aggregate_batch_results(result_payloads, bootstrap_seed=BOOTSTRAP_SEED, n_bootstrap=N_BOOTSTRAP)
summary_rows = []
for method_id, endpoints in aggregate['method_summaries'].items():
    for endpoint, stats in endpoints.items():
        summary_rows.append({'method_id': method_id, 'endpoint': endpoint, **stats})
method_summary_df = pd.DataFrame(summary_rows)
per_run_rows = []
for result in result_payloads:
    if result.get('status') == 'completed_descriptive_only':
        for method_id in METHOD_ORDER:
            per_run_rows.append({'dataset_id': result['inputs']['dataset_id'], 'status': result['status'], 'method_id': method_id, **result['clustering'][method_id]['metrics']})
    else:
        per_run_rows.append({'dataset_id': result.get('inputs', {}).get('dataset_id'), 'status': result.get('status'), 'method_id': None, 'error': result.get('error')})
pd.DataFrame(per_run_rows).to_csv(ARTIFACT_DIR / 'per_run_metrics.csv', index=False)
display(method_summary_df)
method_summary_df.to_csv(ARTIFACT_DIR / 'method_summary.csv', index=False)
(ARTIFACT_DIR / 'aggregate_results.json').write_text(json.dumps(aggregate, ensure_ascii=False, indent=2, default=str) + '\n')

## 17 — Paired Product − Additive bootstrap CI

Đơn vị bootstrap là run. Mỗi delta là `Product − Additive`; CI percentile 95% dùng seed cố định. Không sinh p-value và không dùng CI này cho claim confirmatory. Cần đọc hướng metric: với ARI cao hơn là tốt hơn, còn false destination/review/merge/split thấp hơn là tốt hơn.

In [ ]:
paired_rows = []
for endpoint, values in aggregate['paired_product_minus_additive'].items():
    paired_rows.append({'endpoint': endpoint, 'direction': 'higher_is_better' if endpoint == 'ari_linked' else 'lower_is_better', **{k: v for k, v in values.items() if k != 'deltas'}})
paired_df = pd.DataFrame(paired_rows)
display(paired_df)
paired_df.to_csv(ARTIFACT_DIR / 'paired_product_additive.csv', index=False)

## 18 — Coordinated fake campaign

Mỗi run có ba campaign × 20 report, tức 240 campaign trong full batch. Cell này giữ lại số report bị reject, số destination vận hành và mixed-noise absorption cho từng method; campaign không bị coi là incident thật.

In [ ]:
campaign_rows = aggregate['campaign_rows']
campaign_df = pd.DataFrame(campaign_rows)
if not campaign_df.empty:
    campaign_summary_df = campaign_df.groupby('method_id').agg(
        n_campaigns=('campaign_id', 'nunique'),
        n_reports=('n_reports', 'sum'),
        n_rejected=('n_rejected', 'sum'),
        mean_operational_destinations=('n_operational_destinations', 'mean'),
        campaigns_with_operational_destination=('n_operational_destinations', lambda s: int((s > 0).sum())),
        operational_campaign_rate=('n_operational_destinations', lambda s: float((s > 0).mean())),
        mean_mixed_noise_reports=('mixed_noise_reports', 'mean'),
        mean_mixed_noise_absorption=('mixed_noise_absorption_rate', 'mean'),
    ).reset_index()
else:
    campaign_summary_df = pd.DataFrame()
display(campaign_summary_df)
campaign_df.to_csv(ARTIFACT_DIR / 'campaign_summary.csv', index=False)

## 19 — Deduplication

Cell này báo cáo contract match của exact/near labels và pairwise TP/FP/FN. Exact labels lệch timestamp vẫn được giữ nguyên như adverse quality finding; không sửa timestamp rồi chạy lại trong notebook này.

In [ ]:
dedup_df = pd.DataFrame(aggregate['dedup_rows'])
dedup_df.to_csv(ARTIFACT_DIR / 'dedup_summary.csv', index=False)
dedup_totals = {
    'declared_exact_rows': int(dedup_df['declared_exact_rows'].sum()),
    'exact_fingerprint_matches': int(dedup_df['declared_exact_fingerprint_matches'].sum()),
    'declared_near_rows': int(dedup_df['declared_near_rows'].sum()),
    'near_envelope_matches': int(dedup_df['declared_near_rows_matching_near_envelope'].sum()),
    'true_positive': int(dedup_df['pairwise'].map(lambda x: x['true_positive']).sum()),
    'false_positive': int(dedup_df['pairwise'].map(lambda x: x['false_positive']).sum()),
    'false_negative': int(dedup_df['pairwise'].map(lambda x: x['false_negative']).sum()),
}
dedup_totals['precision'] = dedup_totals['true_positive'] / max(1, dedup_totals['true_positive'] + dedup_totals['false_positive'])
dedup_totals['recall'] = dedup_totals['true_positive'] / max(1, dedup_totals['true_positive'] + dedup_totals['false_negative'])
dedup_totals['f1'] = 2 * dedup_totals['precision'] * dedup_totals['recall'] / max(1e-12, dedup_totals['precision'] + dedup_totals['recall'])
dedup_totals['exact_contract_match_rate'] = dedup_totals['exact_fingerprint_matches'] / max(1, dedup_totals['declared_exact_rows'])
dedup_totals['near_contract_match_rate'] = dedup_totals['near_envelope_matches'] / max(1, dedup_totals['declared_near_rows'])
(ARTIFACT_DIR / 'dedup_pooled_summary.json').write_text(json.dumps(dedup_totals, ensure_ascii=False, indent=2) + '\n')
display(pd.DataFrame([{'dataset_id': r['dataset_id'], **r['pairwise'], 'exact_contract_match_rate': r['declared_exact_fingerprint_matches'] / max(1, r['declared_exact_rows']), 'near_contract_match_rate': r['declared_near_rows_matching_near_envelope'] / max(1, r['declared_near_rows'])} for r in aggregate['dedup_rows']]))
print(dedup_totals)

## 20 — Biểu đồ kết quả

Xuất các hình dùng cho audit: endpoint distribution, paired delta, campaign rejection và dedup contract. Hình chỉ trực quan hóa kết quả đã tính, không chạy thêm một lựa chọn tham số nào.

In [ ]:
FIG_DIR = ARTIFACT_DIR / 'figures'
FIG_DIR.mkdir(exist_ok=True)
plot_endpoints = ['ari_linked', 'false_destinations_per_100_reports', 'noise_rejection', 'review_items_per_100_reports']
for endpoint in plot_endpoints:
    fig, ax = plt.subplots(figsize=(8, 4.5))
    data = []
    labels = []
    for method_id in METHOD_ORDER:
        data.append([float(r['clustering'][method_id]['metrics'][endpoint]) for r in successful])
        labels.append(method_id.replace('method.', ''))
    ax.boxplot(data, labels=labels, showfliers=True)
    ax.set_title(endpoint)
    ax.tick_params(axis='x', rotation=25)
    fig.tight_layout()
    fig.savefig(FIG_DIR / f'{endpoint}.png', dpi=180)
    fig.savefig(FIG_DIR / f'{endpoint}.pdf')
    plt.close(fig)
delta = aggregate['paired_product_minus_additive']['ari_linked']['deltas']
fig, ax = plt.subplots(figsize=(8, 4))
ax.axhline(0, color='black', lw=.8)
ax.plot(range(1, len(delta) + 1), delta, marker='.', lw=.7)
ax.set(xlabel='successful run index', ylabel='Product − Additive ARI', title='Paired ARI deltas')
fig.tight_layout(); fig.savefig(FIG_DIR / 'paired_ari_delta.png', dpi=180); fig.savefig(FIG_DIR / 'paired_ari_delta.pdf'); plt.close(fig)
if not campaign_df.empty:
    campaign_plot = campaign_df.groupby('method_id').agg(rejected=('n_rejected', 'sum'), reports=('n_reports', 'sum')).reset_index()
    campaign_plot['rejection_rate'] = campaign_plot['rejected'] / campaign_plot['reports']
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(campaign_plot['method_id'].str.replace('method.', '', regex=False), campaign_plot['rejection_rate'])
    ax.set(ylabel='campaign reports rejected / reports', title='Coordinated campaign rejection')
    ax.tick_params(axis='x', rotation=25); fig.tight_layout(); fig.savefig(FIG_DIR / 'campaign_rejection.png', dpi=180); fig.savefig(FIG_DIR / 'campaign_rejection.pdf'); plt.close(fig)
if not dedup_df.empty:
    dedup_plot = pd.DataFrame([{'dataset_id': r['dataset_id'], 'precision': r['pairwise']['precision'], 'recall': r['pairwise']['recall'], 'exact_match_rate': r['declared_exact_fingerprint_matches'] / max(1, r['declared_exact_rows']), 'near_match_rate': r['declared_near_rows_matching_near_envelope'] / max(1, r['declared_near_rows'])} for r in aggregate['dedup_rows']])
    fig, ax = plt.subplots(figsize=(7, 4))
    dedup_plot[['precision', 'recall', 'exact_match_rate', 'near_match_rate']].mean().plot.bar(ax=ax)
    ax.set_ylim(0, 1); ax.set_ylabel('rate'); ax.set_title('Dedup contract and pairwise rates')
    fig.tight_layout(); fig.savefig(FIG_DIR / 'dedup_rates.png', dpi=180); fig.savefig(FIG_DIR / 'dedup_rates.pdf'); plt.close(fig)
composition = reports_df.groupby(['dataset_id', 'report_class']).size().unstack(fill_value=0)
ax = composition.plot(kind='bar', stacked=True, figsize=(14, 4), width=.9)
ax.set(xlabel='dataset_id', ylabel='report count', title='Data composition per run'); ax.tick_params(axis='x', labelrotation=90)
ax.figure.tight_layout(); ax.figure.savefig(FIG_DIR / 'data_composition_per_run.png', dpi=180); ax.figure.savefig(FIG_DIR / 'data_composition_per_run.pdf'); plt.close(ax.figure)
print('figures:', sorted(p.name for p in FIG_DIR.iterdir()))

## 21 — Diễn giải tự động có kiểm soát

Các câu dưới đây chỉ là mô tả định lượng từ artifact. Cảnh báo khoa học luôn được in cùng kết quả: ARI cao không chứng minh fake safety; benchmark không có outcome độc lập cho priority/dispatch; và không được gọi là real-world validation.

In [ ]:
best_ari = method_summary_df[method_summary_df['endpoint'] == 'ari_linked'].sort_values('mean', ascending=False).iloc[0]
fake_rows = campaign_summary_df if not campaign_summary_df.empty else pd.DataFrame()
print(f'- Method có mean ARI cao nhất trong batch mô tả: {best_ari["method_id"]} ({best_ari["mean"]:.4f}).')
if not fake_rows.empty:
    print('- Campaign analysis vẫn được giữ riêng; hãy đọc n_rejected và mean_operational_destinations, không diễn giải fake thành incident.')
print('- Đây là exploratory descriptive evidence trên semi-synthetic data; không có p-value hay claim confirmatory.')
print('- Không suy ra priority alignment, dispatch benefit, harm reduction hoặc real-world rescue validity.')
print('- Adverse results và quality violations phải được giữ nguyên trong bài phân tích tiếp theo.')

## 22 — Đóng gói artifact và checksum

Cell này tạo `SHA256SUMS`, manifest liệt kê mọi file và ZIP kết quả. ZIP không bao gồm chính nó để checksum không bị vòng lặp. Nếu full batch có failure, trạng thái đó vẫn nằm trong manifest và tên file không bị đổi thành success.

In [ ]:
import hashlib, zipfile
def sha256_path(path):
    h = hashlib.sha256()
    with open(path, 'rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            h.update(block)
    return h.hexdigest()

manifest_files = {}
for path in sorted(p for p in ARTIFACT_DIR.rglob('*') if p.is_file() and p.name not in {'manifest.json', 'SHA256SUMS'}):
    manifest_files[str(path.relative_to(ARTIFACT_DIR))] = sha256_path(path)
(ARTIFACT_DIR / 'SHA256SUMS').write_text(''.join(f'{digest}  {name}\n' for name, digest in sorted(manifest_files.items())))
artifact_manifest = {'schema_version': 'v2.external-sanity-artifact.1', 'provenance': PROVENANCE, 'status': batch_report['status'], 'files': manifest_files}
(ARTIFACT_DIR / 'manifest.json').write_text(json.dumps(artifact_manifest, ensure_ascii=False, indent=2, default=str) + '\n')
zip_path = ARTIFACT_DIR.with_suffix('.zip')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(p for p in ARTIFACT_DIR.rglob('*') if p.is_file()):
        archive.write(path, path.relative_to(ARTIFACT_DIR.parent))
print('ZIP:', zip_path, 'bytes:', zip_path.stat().st_size)

## 23 — Đưa executed notebook vào ZIP và tải về

Colab không luôn lưu notebook đã thực thi thành file trong `/content`. Hãy tải notebook bằng `File → Download → Download .ipynb`, sau đó upload ở cell này nếu muốn bản executed nằm trong ZIP. Cell xác minh hash và tạo lại ZIP sau khi thêm notebook.

In [ ]:
try:
    from google.colab import files
    print('Nếu muốn đưa executed notebook vào ZIP, chọn file .ipynb trong hộp thoại upload.')
    uploaded = files.upload()
    for name, content in uploaded.items():
        if name.endswith('.ipynb'):
            target = ARTIFACT_DIR / 'datav2_external_benchmark_colab.executed.ipynb'
            target.write_bytes(content)
            print('Saved executed notebook:', target, sha256_path(target))
except Exception as exc:
    print('Không upload được notebook; hãy copy notebook đã chạy vào artifact directory thủ công.', exc)

manifest_files = {}
for path in sorted(p for p in ARTIFACT_DIR.rglob('*') if p.is_file() and p.name not in {'manifest.json', 'SHA256SUMS'}):
    manifest_files[str(path.relative_to(ARTIFACT_DIR))] = sha256_path(path)
(ARTIFACT_DIR / 'SHA256SUMS').write_text(''.join(f'{digest}  {name}\n' for name, digest in sorted(manifest_files.items())))
artifact_manifest['files'] = manifest_files
(ARTIFACT_DIR / 'manifest.json').write_text(json.dumps(artifact_manifest, ensure_ascii=False, indent=2, default=str) + '\n')
with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for path in sorted(p for p in ARTIFACT_DIR.rglob('*') if p.is_file()):
        archive.write(path, path.relative_to(ARTIFACT_DIR.parent))
print('Final ZIP:', zip_path, 'SHA256:', sha256_path(zip_path))
try:
    files.download(str(zip_path))
except Exception as exc:
    print('Tải thủ công file này:', zip_path, exc)